In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from scipy.special import voigt_profile
from scipy.optimize import brentq
from IPython.display import display


# ============================================================
# ATOMIC LINEWIDTH — LASER-FREQUENCY AXIS
# ============================================================

TAU_E = 238.5e-9

W_ATOM_SUM_HZ = 1 / (2 * np.pi * TAU_E)
W_ATOM_LASER_HZ = W_ATOM_SUM_HZ / 2

# Work in MHz throughout the profile calculations.
W_ATOM = W_ATOM_LASER_HZ / 1e6


def lorentzian(x, fwhm):
    """Area-normalized Lorentzian; x and fwhm are in MHz."""
    gamma = fwhm / 2
    return (gamma / np.pi) / (x**2 + gamma**2)


def calculate_fwhm(profile, initial_bound):
    """FWHM of a symmetric profile centred at zero, in MHz."""
    half = float(profile(0)) / 2
    upper = initial_bound

    while float(profile(upper)) > half:
        upper *= 2

    x_half = brentq(
        lambda xx: float(profile(xx)) - half,
        0,
        upper
    )
    return 2 * x_half


def update_plot(laser_fwhm_kHz, phase_diffusion):
    # Original single-photon laser FWHM at 778 nm.
    W_laser = laser_fwhm_kHz / 1000  # kHz -> MHz

    # --------------------------------------------------------
    # Gaussian laser jitter * Lorentzian atomic resonance
    # --------------------------------------------------------
    sigma = W_laser / (2 * np.sqrt(2 * np.log(2)))

    def gaussian_plus_atom(x):
        return voigt_profile(x, sigma, W_ATOM / 2)

    # --------------------------------------------------------
    # Lorentzian contribution * Lorentzian atomic resonance
    # --------------------------------------------------------
    factor = 2 if phase_diffusion else 1
    W_lorentz_contribution = factor * W_laser

    def lorentzian_plus_atom(x):
        return lorentzian(x, W_ATOM + W_lorentz_contribution)

    W_GL = calculate_fwhm(
        gaussian_plus_atom,
        W_ATOM + W_laser
    )
    W_LL = W_ATOM + W_lorentz_contribution

    # Dynamically expand the frequency range.
    span = 3 * max(W_GL, W_LL)
    x = np.linspace(-span, span, 12001)

    fig, ax = plt.subplots(
        figsize=(10, 5),
        layout="constrained"
    )

    # Show the atomic line as a reference.
    ax.plot(
        x,
        lorentzian(x, W_ATOM) / lorentzian(0, W_ATOM),
        color="0.5",
        linestyle=":",
        label=f"Natural atomic line: {W_ATOM * 1000:.1f} kHz"
    )

    lorentz_label = (
        "Atomic + two-photon phase diffusion"
        if phase_diffusion
        else "Atomic Lorentzian * Lorentzian kernel"
    )

    curves = [
        (
            gaussian_plus_atom,
            W_GL,
            "Atomic Lorentzian * Gaussian jitter",
            "C0"
        ),
        (
            lorentzian_plus_atom,
            W_LL,
            lorentz_label,
            "C1"
        ),
    ]

    for profile, width, label, color in curves:
        # Peak normalization is only for displaying the shapes.
        y = profile(x) / profile(0)

        ax.plot(
            x, y,
            color=color,
            linewidth=2,
            label=f"{label}: FWHM = {width * 1000:.1f} kHz"
        )

        # Mark both half-height crossings.
        ax.plot(
            [-width / 2, width / 2],
            [0.5, 0.5],
            color=color,
            marker="|",
            markersize=14,
            linewidth=2
        )

    ax.axhline(0.5, color="0.6", linestyle="--", linewidth=1)

    ax.set(
        xlabel="Laser-frequency detuning from resonance (MHz)",
        ylabel="Peak-normalized fluorescence",
        ylim=(0, 1.05),
        title=(
            f"Original laser FWHM = {laser_fwhm_kHz:.0f} kHz\n"
            f"Atomic FWHM on laser axis = {W_ATOM * 1000:.1f} kHz"
        )
    )

    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
    plt.show()
    plt.close(fig)

    print(f"Original laser FWHM:           {laser_fwhm_kHz:.1f} kHz")
    print(f"Gaussian kernel FWHM:          {W_laser * 1000:.1f} kHz")
    print(
        f"Lorentzian contribution FWHM:  "
        f"{W_lorentz_contribution * 1000:.1f} kHz"
    )


# ============================================================
# INTERACTIVE CONTROLS
# ============================================================

laser_slider = widgets.FloatSlider(
    value=600,
    min=0,
    max=2000,
    step=10,
    description="Laser FWHM (kHz)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px"),
    continuous_update=False
)

phase_checkbox = widgets.Checkbox(
    value=False,
    description="Apply same-laser two-photon phase diffusion (2 × laser FWHM)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto")
)

output = widgets.interactive_output(
    update_plot,
    {
        "laser_fwhm_kHz": laser_slider,
        "phase_diffusion": phase_checkbox
    }
)

display(
    widgets.VBox([laser_slider, phase_checkbox]),
    output
)

Output()